# Day 01: Medallion Architecture Fundamentals (Bronze, Silver & Gold Ingestion)
> **Part 1 of 10 in the PySpark Production Data Engineering Series**

[![PySpark](https://img.shields.io/badge/PySpark-3.5%2B%20%2F%204.0%2B-F26522?style=for-the-badge&logo=apachespark&logoColor=white)](https://spark.apache.org/)
[![Python](https://img.shields.io/badge/Python-3.10%2B-3776AB?style=for-the-badge&logo=python&logoColor=white)](https://python.org)
[![Status](https://img.shields.io/badge/Module-Day%2001%20of%2010-008080?style=for-the-badge)](../README.md)

---

## 📦 Day 01: Medallion Architecture Fundamentals (Bronze, Silver & Gold Ingestion)
### 🎯 Learning Objectives:
1. Initialize a production-ready **SparkSession** with proper Hadoop & Windows compatibility (`winutils`).
2. Implement **Explicit Schema Enforcement** using `StructType` and `StructField` to prevent corrupt data loading.
3. Establish the **Medallion Architecture**:
   - **Bronze Layer**: Raw, immutable, append-only ingested Parquet data.
   - **Silver Layer**: Cleansed, typed, deduplicated, null-handled data with calculated business fields.
   - **Gold Layer**: Aggregated, query-ready analytical marts (by category, by city).
4. Persist each layer efficiently using Apache Parquet columnar storage format.
---


%pip install winutils-pyspark


In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
print("PySpark imported successfully")

PySpark imported successfully


In [3]:
import os
from pyspark.sql import SparkSession

# 1. Configure Hadoop environment with downloaded winutils.exe & hadoop.dll
dummy_hadoop_dir = os.path.abspath("./hadoop_dummy")
os.makedirs(os.path.join(dummy_hadoop_dir, "bin"), exist_ok=True)
os.environ["HADOOP_HOME"] = dummy_hadoop_dir
os.environ["PATH"] = os.path.join(dummy_hadoop_dir, "bin") + ";" + os.environ.get("PATH", "")

# 2. Build SparkSession with RawLocalFileSystem configured
spark = SparkSession.builder \
    .appName("BronzePipeline") \
    .config("spark.hadoop.fs.file.impl", "org.apache.hadoop.fs.RawLocalFileSystem") \
    .getOrCreate()


In [4]:
spark=(
    SparkSession.builder
    .appName("EcommerceDataEngineering")
    .master("local[*]")
    .getOrCreate()
)
spark

In [5]:
import pandas as pd

In [6]:
data = {
    "order_id": [
        1001, 1002, 1003, 1004, 1005,
        1006, 1007, 1008, 1008, 1009
    ],

    "customer_id": [
        101, 102, 103, 101, 104,
        105, 106, 102, 102, 107
    ],

    "customer_name": [
        "Rahul", "Amit", "Sneha", "Rahul", "Priya",
        "Arjun", "Riya", "Amit", "Amit", "Karan"
    ],

    "city": [
        "Kolkata", "Delhi", "Mumbai", "Kolkata", "Pune",
        "Bangalore", "Chennai", "Delhi", "Delhi", None
    ],

    "product": [
        "Laptop", "Mouse", "Keyboard", "Monitor", "Laptop",
        "Mouse", "Keyboard", "Laptop", "Laptop", "Monitor"
    ],

    "category": [
        "Electronics", "Accessories", "Accessories", "Electronics",
        "Electronics", "Accessories", "Accessories", "Electronics",
        "Electronics", "Electronics"
    ],

    "quantity": [
        1, 2, 1, 1, 2,
        3, 2, 1, 1, None
    ],

    "price": [
        55000, 800, 1500, 12000, 55000,
        800, 1500, 55000, 55000, 12000
    ],

    "order_date": [
        "2026-09-01",
        "2026-09-02",
        "2026-09-02",
        "2026-09-03",
        "2026-09-04",
        "2026-09-05",
        "2026-09-06",
        "2026-09-07",
        "2026-09-07",
        "2026-09-08"
    ]
}

pdf = pd.DataFrame(data)

pdf

,order_id,customer_id,customer_name,city,product,category,quantity,price,order_date
0,1001,101,Rahul,Kolkata,Laptop,Electronics,1.0,55000,2026-09-01
1,1002,102,Amit,Delhi,Mouse,Accessories,2.0,800,2026-09-02
2,1003,103,Sneha,Mumbai,Keyboard,Accessories,1.0,1500,2026-09-02
3,1004,101,Rahul,Kolkata,Monitor,Electronics,1.0,12000,2026-09-03
4,1005,104,Priya,Pune,Laptop,Electronics,2.0,55000,2026-09-04
5,1006,105,Arjun,Bangalore,Mouse,Accessories,3.0,800,2026-09-05
6,1007,106,Riya,Chennai,Keyboard,Accessories,2.0,1500,2026-09-06
7,1008,102,Amit,Delhi,Laptop,Electronics,1.0,55000,2026-09-07
8,1008,102,Amit,Delhi,Laptop,Electronics,1.0,55000,2026-09-07
9,1009,107,Karan,None,Monitor,Electronics,NaN,12000,2026-09-08


In [7]:
pdf.to_csv(
    "orders.csv",
    index=False
)
print("CSV created successfully")

CSV created successfully


In [8]:
orders_df=(
    spark.read
    .option("header",True)
    .option("inferSchema",True)
    .csv("orders.csv")
)

In [9]:
orders_df.show()

+--------+-----------+-------------+---------+--------+-----------+--------+-----+----------+
|order_id|customer_id|customer_name|     city| product|   category|quantity|price|order_date|
+--------+-----------+-------------+---------+--------+-----------+--------+-----+----------+
|    1001|        101|        Rahul|  Kolkata|  Laptop|Electronics|     1.0|55000|2026-09-01|
|    1002|        102|         Amit|    Delhi|   Mouse|Accessories|     2.0|  800|2026-09-02|
|    1003|        103|        Sneha|   Mumbai|Keyboard|Accessories|     1.0| 1500|2026-09-02|
|    1004|        101|        Rahul|  Kolkata| Monitor|Electronics|     1.0|12000|2026-09-03|
|    1005|        104|        Priya|     Pune|  Laptop|Electronics|     2.0|55000|2026-09-04|
|    1006|        105|        Arjun|Bangalore|   Mouse|Accessories|     3.0|  800|2026-09-05|
|    1007|        106|         Riya|  Chennai|Keyboard|Accessories|     2.0| 1500|2026-09-06|
|    1008|        102|         Amit|    Delhi|  Laptop|Elect

In [10]:
orders_df.printSchema()

root
 |-- order_id: integer (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- product: string (nullable = true)
 |-- category: string (nullable = true)
 |-- quantity: double (nullable = true)
 |-- price: integer (nullable = true)
 |-- order_date: date (nullable = true)



In [11]:
order_schema = StructType([
    StructField("order_id", IntegerType(), False),
    StructField("customer_id", IntegerType(), False),
    StructField("customer_name", StringType(), True),
    StructField("city", StringType(), True),
    StructField("product", StringType(), True),
    StructField("category", StringType(), True),
    StructField("quantity", IntegerType(), True),
    StructField("price", DoubleType(), True),
    StructField("order_date", StringType(), True)
])

In [12]:
orders_df.show()

+--------+-----------+-------------+---------+--------+-----------+--------+-----+----------+
|order_id|customer_id|customer_name|     city| product|   category|quantity|price|order_date|
+--------+-----------+-------------+---------+--------+-----------+--------+-----+----------+
|    1001|        101|        Rahul|  Kolkata|  Laptop|Electronics|     1.0|55000|2026-09-01|
|    1002|        102|         Amit|    Delhi|   Mouse|Accessories|     2.0|  800|2026-09-02|
|    1003|        103|        Sneha|   Mumbai|Keyboard|Accessories|     1.0| 1500|2026-09-02|
|    1004|        101|        Rahul|  Kolkata| Monitor|Electronics|     1.0|12000|2026-09-03|
|    1005|        104|        Priya|     Pune|  Laptop|Electronics|     2.0|55000|2026-09-04|
|    1006|        105|        Arjun|Bangalore|   Mouse|Accessories|     3.0|  800|2026-09-05|
|    1007|        106|         Riya|  Chennai|Keyboard|Accessories|     2.0| 1500|2026-09-06|
|    1008|        102|         Amit|    Delhi|  Laptop|Elect

In [13]:
orders_df = (
    spark.read
    .option("header", True)
    .schema(order_schema)
    .csv("orders.csv")
)

orders_df.printSchema()

root
 |-- order_id: integer (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- product: string (nullable = true)
 |-- category: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- price: double (nullable = true)
 |-- order_date: string (nullable = true)



In [14]:
print("Rows:",orders_df.count())
print("Columns:",len(orders_df.columns))

Rows: 10
Columns: 9


In [15]:
orders_df.columns

['order_id',
 'customer_id',
 'customer_name',
 'city',
 'product',
 'category',
 'quantity',
 'price',
 'order_date']

In [16]:
orders_df.show(5,truncate=False)

+--------+-----------+-------------+-------+--------+-----------+--------+-------+----------+
|order_id|customer_id|customer_name|city   |product |category   |quantity|price  |order_date|
+--------+-----------+-------------+-------+--------+-----------+--------+-------+----------+
|1001    |101        |Rahul        |Kolkata|Laptop  |Electronics|NULL    |55000.0|2026-09-01|
|1002    |102        |Amit         |Delhi  |Mouse   |Accessories|NULL    |800.0  |2026-09-02|
|1003    |103        |Sneha        |Mumbai |Keyboard|Accessories|NULL    |1500.0 |2026-09-02|
|1004    |101        |Rahul        |Kolkata|Monitor |Electronics|NULL    |12000.0|2026-09-03|
|1005    |104        |Priya        |Pune   |Laptop  |Electronics|NULL    |55000.0|2026-09-04|
+--------+-----------+-------------+-------+--------+-----------+--------+-------+----------+
only showing top 5 rows


In [17]:
# Checking Null Values
orders_df.select([
    count(
        when(col(c).isNull(),c)
    ).alias(c)
    for c in orders_df.columns
]).show()

+--------+-----------+-------------+----+-------+--------+--------+-----+----------+
|order_id|customer_id|customer_name|city|product|category|quantity|price|order_date|
+--------+-----------+-------------+----+-------+--------+--------+-----+----------+
|       0|          0|            0|   1|      0|       0|      10|    0|         0|
+--------+-----------+-------------+----+-------+--------+--------+-----+----------+



In [18]:
#Checking Duplicates
print("Total rows:",orders_df.count())
print(
"Distinct rows:",
    orders_df.dropDuplicates().count()
)

Total rows: 10
Distinct rows: 9


In [19]:
orders_df.groupBy("order_id")\
.count()\
.filter(col("count")>1)\
.show()

+--------+-----+
|order_id|count|
+--------+-----+
|    1008|    2|
+--------+-----+



In [20]:
#Bronze Layer
bronze_df=orders_df

In [21]:
bronze_df.write \
    .mode("overwrite") \
    .parquet("bronze_orders")

In [22]:
bronze_df=spark.read.parquet("bronze_orders")
bronze_df.show()

+--------+-----------+-------------+---------+--------+-----------+--------+-------+----------+
|order_id|customer_id|customer_name|     city| product|   category|quantity|  price|order_date|
+--------+-----------+-------------+---------+--------+-----------+--------+-------+----------+
|    1001|        101|        Rahul|  Kolkata|  Laptop|Electronics|    NULL|55000.0|2026-09-01|
|    1002|        102|         Amit|    Delhi|   Mouse|Accessories|    NULL|  800.0|2026-09-02|
|    1003|        103|        Sneha|   Mumbai|Keyboard|Accessories|    NULL| 1500.0|2026-09-02|
|    1004|        101|        Rahul|  Kolkata| Monitor|Electronics|    NULL|12000.0|2026-09-03|
|    1005|        104|        Priya|     Pune|  Laptop|Electronics|    NULL|55000.0|2026-09-04|
|    1006|        105|        Arjun|Bangalore|   Mouse|Accessories|    NULL|  800.0|2026-09-05|
|    1007|        106|         Riya|  Chennai|Keyboard|Accessories|    NULL| 1500.0|2026-09-06|
|    1008|        102|         Amit|    

In [23]:
#Clean the data
silver_df=bronze_df.dropDuplicates(["order_id"])

In [24]:
silver_df.count()

9

In [25]:
#Handle NULL quantity
silver_df=silver_df.fillna({
    "quantity":1
})

In [26]:
silver_df.select(
    "order_id",
    "quantity"
).show()

+--------+--------+
|order_id|quantity|
+--------+--------+
|    1005|       1|
|    1008|       1|
|    1002|       1|
|    1001|       1|
|    1006|       1|
|    1007|       1|
|    1003|       1|
|    1004|       1|
|    1009|       1|
+--------+--------+



In [27]:
#Handle Null City
silver_df=silver_df.fillna({
    "city":"Unknown"
})

In [28]:
silver_df=silver_df.withColumn(
    "order_date",
    to_date(col("order_date"))
)

In [29]:
silver_df.printSchema()

root
 |-- order_id: integer (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- city: string (nullable = false)
 |-- product: string (nullable = true)
 |-- category: string (nullable = true)
 |-- quantity: integer (nullable = false)
 |-- price: double (nullable = true)
 |-- order_date: date (nullable = true)



In [30]:
#Calculate revenue
silver_df=silver_df.withColumn(
    "revenue",
    col("quantity")*col("price")
)

In [31]:
silver_df.select(
    "order_id",
    "product",
    "quantity",
    "price",
    "revenue"
).show()

+--------+--------+--------+-------+-------+
|order_id| product|quantity|  price|revenue|
+--------+--------+--------+-------+-------+
|    1001|  Laptop|       1|55000.0|55000.0|
|    1002|   Mouse|       1|  800.0|  800.0|
|    1003|Keyboard|       1| 1500.0| 1500.0|
|    1004| Monitor|       1|12000.0|12000.0|
|    1005|  Laptop|       1|55000.0|55000.0|
|    1006|   Mouse|       1|  800.0|  800.0|
|    1007|Keyboard|       1| 1500.0| 1500.0|
|    1008|  Laptop|       1|55000.0|55000.0|
|    1009| Monitor|       1|12000.0|12000.0|
+--------+--------+--------+-------+-------+



In [32]:
silver_df.write\
.mode("overwrite")\
.parquet("silver_orders")

In [33]:
#Gold dataset
from pyspark.sql.functions import countDistinct, sum
gold_category=(
    silver_df
    .groupBy("category")
    .agg(
        sum("revenue").alias("total_revenue"),
        countDistinct("order_id").alias("total_orders")
    )
)

In [34]:
gold_category.show()

+-----------+-------------+------------+
|   category|total_revenue|total_orders|
+-----------+-------------+------------+
|Electronics|     189000.0|           5|
|Accessories|       4600.0|           4|
+-----------+-------------+------------+



In [35]:
#Gold by City
gold_city = (
    silver_df
    .groupBy("city")
    .agg(
        sum("revenue").alias("total_revenue"),
        countDistinct("customer_id").alias("unique_customers")
    )
    .orderBy(desc("total_revenue"))
)

In [36]:
gold_city.show()

+---------+-------------+----------------+
|     city|total_revenue|unique_customers|
+---------+-------------+----------------+
|  Kolkata|      67000.0|               1|
|    Delhi|      55800.0|               1|
|     Pune|      55000.0|               1|
|  Unknown|      12000.0|               1|
|  Chennai|       1500.0|               1|
|   Mumbai|       1500.0|               1|
|Bangalore|        800.0|               1|
+---------+-------------+----------------+



In [37]:
#Save Gold
gold_category.write \
    .mode("overwrite") \
    .parquet("gold_category")

In [38]:
gold_city.write \
    .mode("overwrite") \
    .parquet("gold_city")

---
### 🎓 Day 01 Complete!
Great job completing this module!
👉 **Next Up:** [**Day 02: Production Data Quality, Quarantine & Validation Framework**](./Day_02_Data_Quality_Validation_and_Quarantine.ipynb)
